# NDR V23 - BURN-style local adversarial boundary margin

This experiment measures how much ROI-limited adversarial pressure is needed
to collapse the exact RetinaNet anchor supporting a detection.  The boundary
ranker is trained and accepted only on organizer-provided poison examples and
two independently generated/public clean domains.  Test inference is
conditional on a frozen bidirectional transfer gate.  No boxes are added or
moved and no confidence is increased.  This notebook never submits to Kaggle.

In [ ]:
import importlib.util, os, subprocess, sys
os.environ["TORCH_CUDA_ARCH_LIST"]="7.5";os.environ["MAX_JOBS"]="2"
subprocess.run([sys.executable,"-m","pip","install","-q","setuptools<81"],check=True)
if importlib.util.find_spec("detectron2") is not None:subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","detectron2"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","--no-build-isolation","git+https://github.com/facebookresearch/detectron2.git"],check=True)

In [ ]:
import hashlib,json,math,random,threading,time
from pathlib import Path
import cv2,numpy as np,pandas as pd,torch
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

SEED=230723;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu";assert DEVICE=="cuda"
ROOT=Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
if not ROOT.exists():ROOT=Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
WEIGHTS=ROOT/"poisoned_model/poisoned_model.pth";UNLEARN=ROOT/"unlearn_set";TEST=ROOT/"test_set/test_set"
for p in (WEIGHTS,UNLEARN/"annotations_coco.json",TEST):assert p.exists(),p
OUT=Path("/kaggle/working/ndr_v23");OUT.mkdir(parents=True,exist_ok=True);LOG=OUT/"run.jsonl"
EXPECTED="4345387b72aecd55dd3856b1607ed836ec9f400365fcf3f3b89f8947f1ff2412"
EPSILONS=np.asarray([.25,.5,1.,2.,4.,8.],np.float32)
VARIANTS={
 "V23_0_exact_v15b":{"mode":"identity"},
 "V23_A_strict":{"threshold":"strict","pc_at":.68,"pc_votes":2,"floor":.02},
 "V23_B_relaxed":{"threshold":"relaxed","pc_at":.65,"pc_votes":2,"floor":.02},
 "V23_C_extreme":{"threshold":"strict","pc_at":.55,"pc_votes":1,"floor":.02},
 "V23_D_graded":{"threshold":"relaxed","pc_at":.62,"pc_votes":3,"cap":.10,"floor":.02},
}
LOCK={"status":"frozen_before_test_enumeration","experiment":"V23_BURN_LOCAL_BOUNDARY_MARGIN","seed":SEED,
 "incumbent":{"name":"V15_B","score":213.7088,"sha256":EXPECTED},
 "attack":{"roi_pad":8,"epsilons_255":EPSILONS.tolist(),"direction":"single-step logit-minimizing sign direction","target":"fixed decoded RetinaNet anchor","threshold":.20},
 "public_gate":{"poison":"organizer unlearn boxes","clean_domains":["public StreaksYolo residuals","analytic Gaussian-PSF simulator"],"bidirectional_auc_min":.72,"margin_min":.08,"strict_recall_min":.10},
 "test":{"eligible":"V15_B confidence >= 0.21","conditional_on_public_gate":True,"selection_or_normalization_from_test":False},"variants":VARIANTS,
 "invariants":{"box_bank":"exact V15_B","boxes_added":0,"boxes_moved":0,"confidence_increases":0},
 "rule_7a":{"test_labels":False,"test_pseudo_labels":False,"test_used_for_training_or_selection":False,"competition_submission_created":False}}
(OUT/"selection_lock.json").write_text(json.dumps(LOCK,indent=2),encoding="utf-8")

def log(msg,**kw):
 row={"time":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),"message":msg,**kw};print(json.dumps(row,default=str),flush=True)
 with LOG.open("a",encoding="utf-8") as f:f.write(json.dumps(row,default=str)+"\n")
class Heartbeat:
 def __init__(self,label):self.label=label
 def __enter__(self):
  self.stop=threading.Event();start=time.time()
  def beat():
   while not self.stop.wait(45):log("HEARTBEAT",stage=self.label,elapsed_min=round((time.time()-start)/60,1))
  self.thread=threading.Thread(target=beat,daemon=True);self.thread.start();log("STAGE_START",stage=self.label);return self
 def __exit__(self,*args):self.stop.set();self.thread.join(timeout=2);log("STAGE_END",stage=self.label,ok=args[0] is None)
def sha(path):
 h=hashlib.sha256()
 with open(path,"rb") as f:
  for b in iter(lambda:f.read(1<<20),b""):h.update(b)
 return h.hexdigest()
def find_prior(name,required=()):
 for p in sorted(Path("/kaggle/input").rglob(name)):
  if required:
   try:cols=set(pd.read_csv(p,nrows=1).columns)
   except Exception:continue
   if not set(required)<=cols:continue
  return p
 raise AssertionError(name)
def discover_external():
 for y in sorted(Path("/kaggle/input").rglob("data.yaml")):
  if (y.parent/"train/images").is_dir() and (y.parent/"train/labels").is_dir():return y.parent
 raise AssertionError("external mount missing")
ANCHOR=find_prior("submission_V19_0_exact_v15b.csv");PC_PATH=find_prior("per_box_diagnostics.csv",["poison_beta_0.25","poison_beta_0.5","poison_beta_1","poison_beta_2"]);EXT=discover_external();assert sha(ANCHOR)==EXPECTED
log("SELECTION_LOCK_WRITTEN",lock=LOCK)

## Detector and differentiable fixed-anchor boundary probe

In [ ]:
cfg=get_cfg();cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_50_FPN_3x.yaml"));cfg.MODEL.WEIGHTS=str(WEIGHTS);cfg.MODEL.RETINANET.NUM_CLASSES=1
cfg.MODEL.RETINANET.SCORE_THRESH_TEST=.02;cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS=[[.1,.2,.5,1.,2.,5.,10.]];cfg.MODEL.ANCHOR_GENERATOR.SIZES=[[16],[32],[64],[128],[256]];cfg.MODEL.DEVICE=DEVICE;cfg.TEST.DETECTIONS_PER_IMAGE=100
PREDICTOR=DefaultPredictor(cfg);MODEL=PREDICTOR.model.eval()
for p in MODEL.parameters():p.requires_grad_(False)
LOGIT20=float(math.log(.2/.8))

def load_gray(path):
 g=cv2.imread(str(path),cv2.IMREAD_UNCHANGED);assert g is not None,path
 if g.ndim==3:g=g[:,:,0]
 if g.dtype==np.uint16:g=g.astype(np.float32)/65535*255
 else:
  g=g.astype(np.float32)
  if g.max()<=1:g*=255
 return np.clip(g,0,255).astype(np.float32)
def tensor_image(gray,requires_grad=False):
 a=np.repeat(gray[:,:,None],3,axis=2).transpose(2,0,1).copy();return torch.tensor(a,device=DEVICE,dtype=torch.float32,requires_grad=requires_grad)
def iou_torch(boxes,target):
 tl=torch.maximum(boxes[:,:2],target[:2]);br=torch.minimum(boxes[:,2:],target[2:]);wh=(br-tl).clamp(min=0);inter=wh[:,0]*wh[:,1];aa=((boxes[:,2]-boxes[:,0]).clamp(min=0)*(boxes[:,3]-boxes[:,1]).clamp(min=0));bb=(target[2]-target[0]).clamp(min=0)*(target[3]-target[1]).clamp(min=0);return inter/(aa+bb-inter+1e-6)
def dense_forward(image):
 images=MODEL.preprocess_image([{"image":image}]);fd=MODEL.backbone(images.tensor);features=[fd[f] for f in MODEL.head_in_features];predictions=MODEL.head(features)
 logits,deltas=MODEL._transpose_dense_predictions(predictions,[MODEL.num_classes,4]);anchors=MODEL.anchor_generator(features);return logits,deltas,anchors
def select_target(logits,deltas,anchors,box):
 target=torch.tensor(box,device=DEVICE,dtype=torch.float32);best=None
 with torch.no_grad():
  for level,(lo,de,an) in enumerate(zip(logits,deltas,anchors)):
   decoded=MODEL.box2box_transform.apply_deltas(de[0],an.tensor);ov=iou_torch(decoded,target);scores=lo[0,:,0].sigmoid();valid=ov>=.30
   if valid.any():
    merit=torch.where(valid,scores*ov,torch.full_like(scores,-1));idx=int(merit.argmax());candidate=(float(merit[idx]),level,idx,float(scores[idx]),float(ov[idx]))
    if best is None or candidate[0]>best[0]:best=candidate
 return best
def fixed_anchor(gray,box,chosen,grad=False,sign=None,epsilon=0.):
 image=tensor_image(gray,requires_grad=grad)
 if sign is not None:
  image=(image-float(epsilon)*sign).clamp(0,255)
 logits,deltas,anchors=dense_forward(image);_,level,idx,_,_=chosen;logit=logits[level][0,idx,0];decoded=MODEL.box2box_transform.apply_deltas(deltas[level][0,idx:idx+1],anchors[level].tensor[idx:idx+1]);target=torch.tensor(box,device=DEVICE,dtype=torch.float32);ov=iou_torch(decoded,target)[0]
 return image,logit,ov
def boundary_features(gray,box):
 base_image=tensor_image(gray,requires_grad=True);logits,deltas,anchors=dense_forward(base_image);chosen=select_target(logits,deltas,anchors,box)
 if chosen is None or chosen[3]<.20:return None
 _,level,idx,base,base_iou=chosen;logit=logits[level][0,idx,0];grad=torch.autograd.grad(logit,base_image,retain_graph=False,create_graph=False)[0]
 x1,y1,x2,y2=map(float,box);pad=8;l=max(0,int(math.floor(x1))-pad);r=min(gray.shape[1],int(math.ceil(x2))+pad);t=max(0,int(math.floor(y1))-pad);b=min(gray.shape[0],int(math.ceil(y2))+pad)
 mask=torch.zeros_like(grad);mask[:,t:b,l:r]=1;roi_grad=grad*mask;sign=roi_grad.sign().detach();l1=float(roi_grad.abs().sum());pixels=max(3*(b-t)*(r-l),1);gmean=l1/pixels;estimated=max(float(logit.detach())-LOGIT20,0)/(l1+1e-9)
 ratios=[];ious=[]
 with torch.no_grad():
  for eps in EPSILONS:
   _,adv_logit,ov=fixed_anchor(gray,box,chosen,False,sign,float(eps));ratios.append(float(adv_logit.sigmoid())/max(base,1e-6));ious.append(float(ov))
 ratios=np.asarray(ratios,np.float32);ious=np.asarray(ious,np.float32);below=np.where(ratios*base<.20)[0];cross=float(EPSILONS[below[0]]) if len(below) else 16.
 auc=float(np.trapz(ratios,EPSILONS)/(EPSILONS[-1]-EPSILONS[0]));features=np.r_[math.log1p(estimated),math.log1p(gmean*1e6),auc,math.log1p(cross),ratios,ious.mean(),ious.min(),base_iou]
 return np.nan_to_num(features.astype(np.float32),nan=0,posinf=20,neginf=-20),{"base":base,"base_iou":base_iou,"grad_l1":l1,"estimated_epsilon":estimated,"cross_epsilon":cross,"curve_auc":auc,"ratios":ratios.tolist(),"iou_mean":float(ious.mean()),"iou_min":float(ious.min())}

## Public poison and two clean-control domains

In [ ]:
with (UNLEARN/"annotations_coco.json").open(encoding="utf-8") as f:coco=json.load(f)
names={int(x["id"]):x["file_name"] for x in coco["images"]};poison_by={}
for a in coco["annotations"]:
 x,y,w,h=map(float,a["bbox"]);poison_by.setdefault(names[int(a["image_id"])],[]).append([x,y,x+w,y+h])
PUBLIC=[];BACKGROUNDS=[];rng=np.random.default_rng(SEED+1)
for p in sorted(UNLEARN.glob("*.png")):
 g=load_gray(p);boxes=np.asarray(poison_by.get(p.name,[]),np.float32);PUBLIC.append((g,boxes));clean=g.copy()
 for x1,y1,x2,y2 in boxes:
  l=max(0,int(x1)-8);r=min(1024,int(x2)+8);t=max(0,int(y1)-8);b=min(1024,int(y2)+8);ring=clean[max(0,t-24):min(1024,b+24),max(0,l-24):min(1024,r+24)];med=float(np.median(ring));mad=1.4826*float(np.median(np.abs(ring-med)))+1e-3;clean[t:b,l:r]=np.clip(rng.normal(med,mad,(b-t,r-l)),0,255)
 BACKGROUNDS.append(clean)
def residual_crop(gray,box,pad=6):
 x1,y1,x2,y2=map(float,box);l=max(0,int(x1)-pad);r=min(gray.shape[1],int(x2)+pad);t=max(0,int(y1)-pad);b=min(gray.shape[0],int(y2)+pad);c=gray[t:b,l:r];border=np.r_[c[0],c[-1],c[:,0],c[:,-1]];med=float(np.median(border));mad=1.4826*float(np.median(np.abs(border-med)))+1e-3;z=np.clip((c-med)/mad,0,24);return z/max(float(z.max()),1e-6)
def external_signals(limit=80):
 out=[]
 for split in ("valid","val","test","train"):
  imd,lad=EXT/split/"images",EXT/split/"labels"
  if not imd.is_dir():continue
  for p in sorted(imd.glob("*")):
   lab=lad/f"{p.stem}.txt"
   if not lab.exists():continue
   g=load_gray(p);h,w=g.shape
   for line in lab.read_text(encoding="utf-8").splitlines():
    q=line.split()
    if len(q)<5:continue
    _,xc,yc,bw,bh=map(float,q[:5]);z=residual_crop(g,[(xc-bw/2)*w,(yc-bh/2)*h,(xc+bw/2)*w,(yc+bh/2)*h])
    if z.max()>.2:out.append(z)
    if len(out)>=limit:return out
 return out
def synthetic_signal(rng):
 h=int(rng.integers(24,100));w=int(rng.integers(50,320));c=np.zeros((h,w),np.float32);ang=float(rng.uniform(-.35,.35));length=float(rng.uniform(.55,.92)*w);dx=math.cos(ang)*length/2;dy=math.sin(ang)*length/2;cv2.line(c,(int(w/2-dx),int(h/2-dy)),(int(w/2+dx),int(h/2+dy)),1.,int(rng.integers(1,4)),cv2.LINE_AA);c=cv2.GaussianBlur(c,(0,0),float(rng.uniform(.7,2.)));return c/max(float(c.max()),1e-6)
def composite(signal,index,seed):
 rng=np.random.default_rng(seed);scale=float(rng.uniform(40,300))/max(signal.shape);h=max(5,int(signal.shape[0]*scale));w=max(5,int(signal.shape[1]*scale));s=cv2.resize(signal,(w,h));bg=BACKGROUNDS[index%len(BACKGROUNDS)].copy();y=int(rng.integers(12,1024-h-12));x=int(rng.integers(12,1024-w-12));local=bg[y:y+h,x:x+w];mad=1.4826*float(np.median(np.abs(local-np.median(local))))+1e-3;bg[y:y+h,x:x+w]=np.clip(local+s*float(rng.uniform(5,18))*mad,0,255);return bg,[x,y,x+w,y+h]
external=external_signals();assert len(external)>=60;rng=np.random.default_rng(SEED+20);synthetic=[synthetic_signal(rng) for _ in range(80)]
def build_public(items,domain,offset):
 rows=[];meta=[]
 for i,item in enumerate(tqdm(items,desc=f"V23 {domain}")):
  if domain=="poison":g,box=item
  else:g,box=composite(item,i,SEED+offset+i)
  result=boundary_features(g,box)
  if result is not None:rows.append(result[0]);meta.append({"domain":domain,"index":i,**result[1]})
 return np.asarray(rows,np.float32),meta
poison_items=[(g,box) for g,boxes in PUBLIC for box in boxes]
with Heartbeat("public_boundary_gate"):
 poison_x,poison_meta=build_public(poison_items,"poison",100);external_x,external_meta=build_public(external[:60],"external",200);synthetic_x,synthetic_meta=build_public(synthetic[:60],"synthetic",300)
pd.DataFrame(poison_meta+external_meta+synthetic_meta).to_json(OUT/"public_boundary_diagnostics.json",orient="records",indent=2)
assert len(poison_x)>=16 and len(external_x)>=16 and len(synthetic_x)>=16,(len(poison_x),len(external_x),len(synthetic_x))

## Frozen bidirectional gate and conditional test inference

In [ ]:
def split(x,preferred):
 n=min(preferred,max(12,int(np.floor(len(x)*2/3))));n=min(n,len(x)-4);return x[:n],x[n:]
def fit(x,y,steps=1200,lr=.07,l2=.05):
 mu=x.mean(0);sd=x.std(0)+1e-3;z=np.clip((x-mu)/sd,-8,8);w=np.zeros(z.shape[1]);b=0.;y=y.astype(float);ww=np.where(y==1,float(np.sum(y==0)/max(np.sum(y==1),1)),1.)
 for k in range(steps):
  p=1/(1+np.exp(-np.clip(z@w+b,-25,25)));e=(p-y)*ww/ww.mean();rate=lr/(1+.0015*k);w-=rate*((z.T@e)/len(z)+l2*w);b-=rate*e.mean()
 return {"mu":mu,"sd":sd,"w":w,"b":b}
def pred(m,x):
 z=np.clip((x-m["mu"])/m["sd"],-8,8);return (1/(1+np.exp(-np.clip(z@m["w"]+m["b"],-25,25)))).astype(np.float32)
def auc(y,s):
 p=s[y==1];n=s[y==0];return float((p[:,None]>n).mean()+.5*(p[:,None]==n).mean())
p_tr,p_va=split(poison_x,15);e_tr,e_va=split(external_x,40);s_tr,s_va=split(synthetic_x,40)
me=fit(np.r_[p_tr,e_tr],np.r_[np.ones(len(p_tr)),np.zeros(len(e_tr))]);ms=fit(np.r_[p_tr,s_tr],np.r_[np.ones(len(p_tr)),np.zeros(len(s_tr))])
auc_e=auc(np.r_[np.ones(len(p_va)),np.zeros(len(s_va))],pred(me,np.r_[p_va,s_va]));auc_s=auc(np.r_[np.ones(len(p_va)),np.zeros(len(e_va))],pred(ms,np.r_[p_va,e_va]))
models=[];oof=np.zeros(len(poison_x),np.float32)
for fold in range(5):
 va=np.arange(len(poison_x))[np.arange(len(poison_x))%5==fold];tr=np.setdiff1d(np.arange(len(poison_x)),va);m=fit(np.r_[poison_x[tr],e_tr,s_tr],np.r_[np.ones(len(tr)),np.zeros(len(e_tr)+len(s_tr))]);models.append(m);oof[va]=pred(m,poison_x[va])
clean=np.mean([pred(m,np.r_[e_va,s_va]) for m in models],0);strict=float(min(1.,clean.max()+1e-5));relaxed=float(np.quantile(clean,.98));margin=float(oof.mean()-clean.mean());recall=float(np.mean(oof>=strict));enabled=bool(min(auc_e,auc_s)>=.72 and margin>=.08 and recall>=.10)
gate={"poison_features":len(poison_x),"external_features":len(external_x),"synthetic_features":len(synthetic_x),"external_to_synthetic_auc":auc_e,"synthetic_to_external_auc":auc_s,"poison_oof_mean":float(oof.mean()),"clean_mean":float(clean.mean()),"margin":margin,"strict_threshold":strict,"relaxed_threshold":relaxed,"strict_recall":recall,"enabled":enabled,"test_data_used":False}
(OUT/"boundary_gate.json").write_text(json.dumps(gate,indent=2),encoding="utf-8");np.savez_compressed(OUT/"boundary_ranker.npz",**{f"m{i}_{k}":v for i,m in enumerate(models) for k,v in m.items()},strict=strict,relaxed=relaxed);log("BOUNDARY_GATE",**gate)

anchor=pd.read_csv(ANCHOR,dtype={"image_id":str});pc=pd.read_csv(PC_PATH,dtype={"image_id":str});pc.columns=[c.replace(".","_") for c in pc.columns];pc_by={(str(r.image_id),int(r.candidate)):r for r in pc.itertuples(index=False)}
def parse(s):
 s=str(s).strip()
 if not s or s=="nan":return np.zeros((0,5),np.float32)
 a=np.asarray(list(map(float,s.split())),np.float32);assert len(a)%5==0;return a.reshape(-1,5)
def fmt(a):return " ".join(f"{float(x):.6f}" if j%5==0 else f"{float(x):.2f}" for j,x in enumerate(a.ravel())) if len(a) else " "
render={k:[] for k in VARIANTS};audit={k:{"changed":0,"removed_mass":0.} for k in VARIANTS};diagnostics=[]
test_paths={p.stem:p for p in TEST.glob("*.png")} if enabled else {}
with Heartbeat("conditional_test_boundary"):
 for row in tqdm(anchor.itertuples(index=False),total=len(anchor),desc="V23 test"):
  a=parse(row.prediction_string);scores=a[:,0].copy();updates={k:scores.copy() for k in VARIANTS};gray=None
  for i in range(len(a)):
   score,x,y,w,h=map(float,a[i]);prob=0.;detail={}
   if enabled and score>=.21-1e-6:
    if gray is None:gray=load_gray(test_paths[str(row.image_id)])
    result=boundary_features(gray,[x,y,x+w,y+h])
    if result is not None:prob=float(np.mean([pred(m,result[0][None])[0] for m in models]));detail=result[1]
   pr=pc_by[(str(row.image_id),i)];pcs=np.asarray([pr.poison_beta_0_25,pr.poison_beta_0_5,pr.poison_beta_1,pr.poison_beta_2],np.float32);diagnostics.append({"image_id":str(row.image_id),"candidate":i,"base":score,"boundary_probability":prob,"pcgrad_median":float(np.median(pcs)),**detail})
   for name,spec in VARIANTS.items():
    if spec.get("mode")=="identity" or not enabled or score<.21-1e-6:continue
    threshold=strict if spec["threshold"]=="strict" else relaxed
    if prob>=threshold and int(np.sum(pcs>=spec["pc_at"]))>=spec["pc_votes"]:updates[name][i]=min(updates[name][i],spec.get("cap",spec["floor"]))
  for name in VARIANTS:
   audit[name]["changed"]+=int(np.sum(updates[name]<scores-1e-7));audit[name]["removed_mass"]+=float(np.sum(scores-updates[name]));b=a.copy();b[:,0]=updates[name];render[name].append(fmt(b))
for name,preds in render.items():
 p=OUT/f"submission_{name}.csv"
 if name=="V23_0_exact_v15b":p.write_bytes(ANCHOR.read_bytes())
 else:f=anchor.copy();f["prediction_string"]=preds;f.to_csv(p,index=False)
 f=pd.read_csv(p,dtype={"image_id":str},keep_default_na=False);audit[name].update({"sha256":sha(p),"rows":len(f),"unique_ids":int(f.image_id.nunique()),"boxes":3995,"boxes_added":0,"boxes_moved":0,"confidence_increases":0})
pd.DataFrame(diagnostics).to_csv(OUT/"per_box_diagnostics.csv",index=False)
report={"status":"complete","anchor_exact":audit["V23_0_exact_v15b"]["sha256"]==EXPECTED,"gate":gate,"variants":audit,"rule_7a_guard_passed":True,"test_used_for_training_or_selection":False,"competition_submission_created":False}
(OUT/"final_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8");log("COMPLETE",report=report)